# Cost and Efficiency Analysis

This notebook loads production 35-task benchmark reports from `logs/` and compares token cost and efficiency by model.

Here, **cost means token consumption**, not USD pricing. Agent model tokens and judge model tokens are kept separate, with combined token totals used only when comparing total benchmark expense.

## Metric Definitions

- **Agent tokens**: input/output/total tokens used by the benchmarked model, read from each task's `query_trace`.
- **Judge tokens**: input/output/total tokens used by the ground-truth text judge, read from task diagnostics when present.
- **Combined tokens**: agent total tokens plus judge total tokens.
- **Tokens per completed task**: total tokens divided by tasks with `status == "completed"`.
- **Tokens per passed task**: total tokens divided by tasks whose ground-truth validation passed.
- **Pass rate per 100k tokens**: passed task-runs divided by combined tokens, scaled to 100,000 tokens.
- **Judge overhead share**: judge tokens divided by combined tokens.

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd

try:
    import matplotlib.pyplot as plt  # noqa: F401
    HAVE_MATPLOTLIB = True
except ModuleNotFoundError:
    HAVE_MATPLOTLIB = False

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## Configuration

In [2]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "analysis" else Path(".").resolve()
LOG_ROOT = PROJECT_ROOT / "logs"
EXPORT_DIR = PROJECT_ROOT / "analysis" / "outputs"

# Default production selection: infer folders that contain three 35-task task_run_report.json files.
# These patterns keep old, pilot, and generated result folders out of cross-model comparisons.
EXCLUDE_FOLDER_PATTERNS = [
    re.compile(r"^results$", re.IGNORECASE),
    re.compile(r"test", re.IGNORECASE),
    re.compile(r"pilot", re.IGNORECASE),
    re.compile(r"30tasks", re.IGNORECASE),
]

EXPECTED_TASKS_PER_RUN = 35
EXPECTED_RUNS_PER_MODEL = 3

# Set to a list like ["gpt5.5", "qwen3.6-plus"] to force a subset.
MODEL_FOLDER_ALLOWLIST: list[str] | None = None

assert LOG_ROOT.exists(), f"Log root not found: {LOG_ROOT}"

## Loader Helpers

In [3]:
def read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def is_excluded_model_folder(path: Path) -> bool:
    return any(pattern.search(path.name) for pattern in EXCLUDE_FOLDER_PATTERNS)


def infer_run_index(run_dir: Path, fallback: int) -> int:
    match = re.match(r"^(\d+)", run_dir.name)
    return int(match.group(1)) if match else fallback


def report_task_count(report_path: Path) -> int | None:
    try:
        report = read_json(report_path)
    except Exception:
        return None
    return len(report.get("results", []))


def production_report_paths(log_root: Path) -> list[Path]:
    selected: list[Path] = []
    for model_dir in sorted(p for p in log_root.iterdir() if p.is_dir()):
        if MODEL_FOLDER_ALLOWLIST is not None and model_dir.name not in MODEL_FOLDER_ALLOWLIST:
            continue
        if is_excluded_model_folder(model_dir):
            continue

        direct_reports = sorted(model_dir.glob("*/task_run_report.json"))
        production_reports = [p for p in direct_reports if report_task_count(p) == EXPECTED_TASKS_PER_RUN]

        if len(production_reports) >= EXPECTED_RUNS_PER_MODEL:
            selected.extend(production_reports)

    return selected


def tokens_from_usage(usage: dict | None) -> dict[str, int | None]:
    usage = usage or {}
    return {
        "input_tokens": usage.get("input_tokens"),
        "output_tokens": usage.get("output_tokens"),
        "total_tokens": usage.get("total_tokens"),
    }


def first_task_set_id(results: list[dict]) -> str | None:
    for result in results:
        query_trace = result.get("query_trace") or {}
        task_set_id = query_trace.get("task_set_id")
        if task_set_id:
            return task_set_id
    return None


report_paths = production_report_paths(LOG_ROOT)
report_paths

[WindowsPath('d:/projects/thesis-code/logs/claude-optus-4.6/anthropic_claude-opus-4.6_20260512T082634Z/task_run_report.json'),
 WindowsPath('d:/projects/thesis-code/logs/claude-optus-4.6/anthropic_claude-opus-4.6_20260512T084016Z/task_run_report.json'),
 WindowsPath('d:/projects/thesis-code/logs/claude-optus-4.6/anthropic_claude-opus-4.6_20260512T085433Z/task_run_report.json'),
 WindowsPath('d:/projects/thesis-code/logs/claude-sonnet-4.6/1anthropic_claude-sonnet-4.6_20260512T072817Z/task_run_report.json'),
 WindowsPath('d:/projects/thesis-code/logs/claude-sonnet-4.6/2anthropic_claude-sonnet-4.6_20260512T073036Z/task_run_report.json'),
 WindowsPath('d:/projects/thesis-code/logs/claude-sonnet-4.6/3anthropic_claude-sonnet-4.6_20260512T073516Z/task_run_report.json'),
 WindowsPath('d:/projects/thesis-code/logs/gpt5.5/openai_gpt-5.5_20260512T075029Z/task_run_report.json'),
 WindowsPath('d:/projects/thesis-code/logs/gpt5.5/openai_gpt-5.5_20260512T075044Z/task_run_report.json'),
 WindowsPath('

## Load Reports

In [4]:
run_rows: list[dict] = []
task_rows: list[dict] = []

for model_folder, folder_reports in pd.Series(report_paths).groupby(lambda i: report_paths[i].parents[1].name):
    sorted_reports = sorted(folder_reports.tolist(), key=lambda p: (infer_run_index(p.parent, 10_000), p.parent.name))

    for fallback_index, report_path in enumerate(sorted_reports, start=1):
        model_dir = report_path.parents[1]
        run_dir = report_path.parent
        run_index = infer_run_index(run_dir, fallback_index)
        report = read_json(report_path)
        summary = report.get("summary", {})
        results = report.get("results", [])
        report_judge_usage = tokens_from_usage(report.get("judge_token_usage") or summary.get("judge_token_usage"))

        model = summary.get("model")
        run_id = run_dir.name

        run_rows.append({
            "model_folder": model_dir.name,
            "run_index": run_index,
            "run_id": run_id,
            "report_path": str(report_path.relative_to(PROJECT_ROOT)),
            "generated_at": report.get("generated_at"),
            "source_csv": report.get("source_csv"),
            "task_set_id": first_task_set_id(results),
            "model": model,
            "total_tasks": summary.get("total_tasks", len(results)),
            "finished_with_result_tasks": summary.get("finished_with_result_tasks"),
            "failed_tasks": summary.get("failed_tasks"),
            "TCR_summary": summary.get("Task Completion Rate (TCR)"),
            "summary_agent_average_total_tokens": summary.get("average_total_tokens"),
            "report_judge_input_tokens": report_judge_usage["input_tokens"],
            "report_judge_output_tokens": report_judge_usage["output_tokens"],
            "report_judge_total_tokens": report_judge_usage["total_tokens"],
        })

        for result in results:
            validation = result.get("ground_truth_validation") or {}
            diagnostics = validation.get("diagnostics") or {}
            judge_usage = tokens_from_usage(diagnostics.get("judge_token_usage"))
            query_trace = result.get("query_trace") or {}
            tool_app = result.get("tool_appropriateness") or {}

            agent_input = query_trace.get("input_tokens")
            agent_output = query_trace.get("output_tokens")
            agent_total = query_trace.get("total_tokens", result.get("total_tokens"))

            task_rows.append({
                "model_folder": model_dir.name,
                "model": model,
                "run_index": run_index,
                "run_id": run_id,
                "task_id": result.get("task_id"),
                "prompt": result.get("prompt"),
                "answer_type": result.get("answer_type"),
                "status": result.get("status"),
                "passed": validation.get("passed", result.get("passed")),
                "ground_truth_score": validation.get("score"),
                "validation_strategy": validation.get("strategy"),
                "agent_input_tokens": agent_input,
                "agent_output_tokens": agent_output,
                "agent_total_tokens": agent_total,
                "judge_input_tokens": judge_usage["input_tokens"],
                "judge_output_tokens": judge_usage["output_tokens"],
                "judge_total_tokens": judge_usage["total_tokens"],
                "latency": result.get("latency"),
                "end_to_end_latency": result.get("end_to_end_latency"),
                "total_steps": result.get("total_steps"),
                "total_tool_calls": result.get("total_tool_calls"),
                "parameter_schema_valid_rate": result.get("parameter_schema_valid_rate"),
                "constraint_compliance_rate": result.get("constraint_compliance_rate"),
                "tool_appropriateness": tool_app.get("average_score"),
                "error_type": result.get("error_type"),
                "error_message": result.get("error_message"),
                "tools_used": query_trace.get("tools_used"),
            })

runs_df = pd.DataFrame(run_rows)
tasks_df = pd.DataFrame(task_rows)

numeric_columns = [
    "total_tasks", "finished_with_result_tasks", "failed_tasks", "TCR_summary",
    "summary_agent_average_total_tokens", "report_judge_input_tokens", "report_judge_output_tokens",
    "report_judge_total_tokens", "ground_truth_score", "agent_input_tokens", "agent_output_tokens",
    "agent_total_tokens", "judge_input_tokens", "judge_output_tokens", "judge_total_tokens", "latency",
    "end_to_end_latency", "total_steps", "total_tool_calls", "parameter_schema_valid_rate",
    "constraint_compliance_rate", "tool_appropriateness",
]
for df in [runs_df, tasks_df]:
    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

tasks_df["passed"] = tasks_df["passed"].astype("boolean")
tasks_df["completed"] = tasks_df["status"].eq("completed")
tasks_df["agent_total_tokens"] = tasks_df["agent_total_tokens"].fillna(0)
tasks_df["judge_total_tokens"] = tasks_df["judge_total_tokens"].fillna(0)
tasks_df["combined_total_tokens"] = tasks_df["agent_total_tokens"] + tasks_df["judge_total_tokens"]

tasks_df.head()

,model_folder,model,run_index,run_id,task_id,prompt,answer_type,status,passed,ground_truth_score,validation_strategy,agent_input_tokens,agent_output_tokens,agent_total_tokens,judge_input_tokens,judge_output_tokens,judge_total_tokens,latency,end_to_end_latency,total_steps,total_tool_calls,parameter_schema_valid_rate,constraint_compliance_rate,tool_appropriateness,error_type,error_message,tools_used,completed,combined_total_tokens
0,claude-optus-4.6,anthropic/claude-opus-4.6,1,anthropic_claude-opus-4.6_20260512T082634Z,A1,how many posts does twitter collection have?,scalar,completed,True,1,scalar_deterministic,"9,183.0000",119.0000,"9,302.0000",NaN,NaN,0.0000,8.9180,8.9312,2.0000,1.0000,1.0000,NaN,1.0000,NaN,NaN,[get_collection_summary],True,"9,302.0000"
1,claude-optus-4.6,anthropic/claude-opus-4.6,1,anthropic_claude-opus-4.6_20260512T082634Z,A2,Compare the harvesting date span of all availa...,text,completed,True,1,text_judge,"14,653.0000","1,170.0000","15,823.0000","1,089.0000",990.0000,"2,079.0000",21.4419,30.8165,3.0000,8.0000,1.0000,NaN,1.0000,NaN,NaN,"[get_collections, get_collection_summary, get_...",True,"17,902.0000"
2,claude-optus-4.6,anthropic/claude-opus-4.6,1,anthropic_claude-opus-4.6_20260512T082634Z,A3,"Compare Reddit, Mastodon, and Twitter by total...",text,completed,True,1,text_judge,"14,220.0000",601.0000,"14,821.0000",963.0000,"1,375.0000","2,338.0000",15.4138,28.6152,3.0000,4.0000,1.0000,NaN,1.0000,NaN,NaN,"[get_collections, get_collection_summary, get_...",True,"17,159.0000"
3,claude-optus-4.6,anthropic/claude-opus-4.6,1,anthropic_claude-opus-4.6_20260512T082634Z,A4,List all available collections and their start...,text,completed,True,1,text_judge,"14,597.0000",596.0000,"15,193.0000",769.0000,"1,424.0000","2,193.0000",12.1675,23.4208,3.0000,8.0000,1.0000,NaN,1.0000,NaN,NaN,"[get_collections, get_collection_summary, get_...",True,"17,386.0000"
4,claude-optus-4.6,anthropic/claude-opus-4.6,1,anthropic_claude-opus-4.6_20260512T082634Z,A5,Which available collections have harvesting da...,text,completed,True,1,text_judge,"14,647.0000",778.0000,"15,425.0000","1,001.0000",839.0000,"1,840.0000",15.4860,25.2906,3.0000,8.0000,1.0000,NaN,1.0000,NaN,NaN,"[get_collections, get_collection_summary, get_...",True,"17,265.0000"


## Data Quality Checks

In [5]:
print(f"Loaded {len(runs_df)} production runs and {len(tasks_df)} task-runs")
display(runs_df.groupby(["model_folder", "model"], dropna=False).agg(runs=("run_id", "nunique"), task_rows=("total_tasks", "sum")).reset_index())

excluded_top_level_folders = sorted(
    p.name for p in LOG_ROOT.iterdir()
    if p.is_dir() and (is_excluded_model_folder(p) or p.name not in runs_df["model_folder"].unique())
)
print("Excluded/non-production top-level folders:", excluded_top_level_folders)

agent_token_mismatch = tasks_df[
    (tasks_df["agent_input_tokens"].fillna(0) + tasks_df["agent_output_tokens"].fillna(0) != tasks_df["agent_total_tokens"].fillna(0))
    & tasks_df["agent_input_tokens"].notna()
    & tasks_df["agent_output_tokens"].notna()
]
print(f"Agent input+output token mismatches: {len(agent_token_mismatch)}")

judge_check = tasks_df.groupby(["model_folder", "run_id"], as_index=False)["judge_total_tokens"].sum().merge(
    runs_df[["model_folder", "run_id", "report_judge_total_tokens"]],
    on=["model_folder", "run_id"],
    how="left",
)
judge_check["judge_token_delta"] = judge_check["judge_total_tokens"] - judge_check["report_judge_total_tokens"].fillna(0)
display(judge_check)

Loaded 18 production runs and 630 task-runs


,model_folder,model,runs,task_rows
0,claude-optus-4.6,anthropic/claude-opus-4.6,3,105
1,claude-sonnet-4.6,anthropic/claude-sonnet-4.6,3,105
2,gpt5.5,openai/gpt-5.5,3,105
3,kimi-k2.6,moonshotai/kimi-k2.6,3,105
4,llama4,meta-llama/llama-4-scout,3,105
5,qwen3.6-plus,qwen/qwen3.6-plus,3,105


Excluded/non-production top-level folders: ['30tasks-qwen3.6-plus', 'results']
Agent input+output token mismatches: 0


,model_folder,run_id,judge_total_tokens,report_judge_total_tokens,judge_token_delta
0,claude-optus-4.6,anthropic_claude-opus-4.6_20260512T082634Z,"55,067.0000",55067,0.0000
1,claude-optus-4.6,anthropic_claude-opus-4.6_20260512T084016Z,"55,721.0000",55721,0.0000
2,claude-optus-4.6,anthropic_claude-opus-4.6_20260512T085433Z,"29,382.0000",29382,0.0000
3,claude-sonnet-4.6,1anthropic_claude-sonnet-4.6_20260512T072817Z,"53,851.0000",53851,0.0000
4,claude-sonnet-4.6,2anthropic_claude-sonnet-4.6_20260512T073036Z,"54,777.0000",54777,0.0000
5,claude-sonnet-4.6,3anthropic_claude-sonnet-4.6_20260512T073516Z,"53,126.0000",53126,0.0000
6,gpt5.5,openai_gpt-5.5_20260512T075029Z,"54,628.0000",54628,0.0000
7,gpt5.5,openai_gpt-5.5_20260512T075044Z,"53,696.0000",53696,0.0000
8,gpt5.5,openai_gpt-5.5_20260512T075850Z,"50,847.0000",50847,0.0000
9,kimi-k2.6,1moonshotai_kimi-k2.6_20260512T065725Z,"51,959.0000",51959,0.0000


## Per-Run Summary

In [6]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator.div(denominator.where(denominator != 0))


run_summary = tasks_df.groupby(["model_folder", "model", "run_index", "run_id"], dropna=False).agg(
    tasks=("task_id", "count"),
    completed_tasks=("completed", "sum"),
    failed_tasks=("completed", lambda s: (~s).sum()),
    passed_tasks=("passed", "sum"),
    agent_total_input_tokens=("agent_input_tokens", "sum"),
    agent_total_output_tokens=("agent_output_tokens", "sum"),
    agent_total_tokens=("agent_total_tokens", "sum"),
    judge_total_tokens=("judge_total_tokens", "sum"),
    combined_total_tokens=("combined_total_tokens", "sum"),
    mean_agent_tokens=("agent_total_tokens", "mean"),
    median_agent_tokens=("agent_total_tokens", "median"),
    mean_judge_tokens=("judge_total_tokens", "mean"),
    average_latency_seconds=("latency", "mean"),
    average_end_to_end_latency_seconds=("end_to_end_latency", "mean"),
    average_tool_calls=("total_tool_calls", "mean"),
    average_steps=("total_steps", "mean"),
).reset_index()

run_summary["pass_rate"] = safe_divide(run_summary["passed_tasks"], run_summary["tasks"])
run_summary["completion_rate"] = safe_divide(run_summary["completed_tasks"], run_summary["tasks"])
run_summary["agent_tokens_per_completed_task"] = safe_divide(run_summary["agent_total_tokens"], run_summary["completed_tasks"])
run_summary["combined_tokens_per_passed_task"] = safe_divide(run_summary["combined_total_tokens"], run_summary["passed_tasks"])
run_summary["judge_overhead_share"] = safe_divide(run_summary["judge_total_tokens"], run_summary["combined_total_tokens"])

display(run_summary.sort_values(["model_folder", "run_index"]))

,model_folder,model,run_index,run_id,tasks,completed_tasks,failed_tasks,passed_tasks,agent_total_input_tokens,agent_total_output_tokens,agent_total_tokens,judge_total_tokens,combined_total_tokens,mean_agent_tokens,median_agent_tokens,mean_judge_tokens,average_latency_seconds,average_end_to_end_latency_seconds,average_tool_calls,average_steps,pass_rate,completion_rate,agent_tokens_per_completed_task,combined_tokens_per_passed_task,judge_overhead_share
0,claude-optus-4.6,anthropic/claude-opus-4.6,1,anthropic_claude-opus-4.6_20260512T082634Z,35,35,0,34,"618,887.0000","40,370.0000","659,257.0000","55,067.0000","714,324.0000","18,835.9143","16,741.0000","1,573.3429",24.1823,31.4983,4.6000,3.0857,0.9714,1.0000,"18,835.9143","21,009.5294",0.0771
1,claude-optus-4.6,anthropic/claude-opus-4.6,2,anthropic_claude-opus-4.6_20260512T084016Z,35,35,0,33,"636,144.0000","43,003.0000","679,147.0000","55,721.0000","734,868.0000","19,404.2000","16,449.0000","1,592.0286",24.2552,31.7438,5.0571,3.1143,0.9429,1.0000,"19,404.2000","22,268.7273",0.0758
2,claude-optus-4.6,anthropic/claude-opus-4.6,3,anthropic_claude-opus-4.6_20260512T085433Z,35,27,8,26,"474,715.0000","28,978.0000","503,693.0000","29,382.0000","533,075.0000","14,391.2286","15,360.0000",839.4857,16.4016,20.3453,5.2593,3.2963,0.7429,0.7714,"18,655.2963","20,502.8846",0.0551
3,claude-sonnet-4.6,anthropic/claude-sonnet-4.6,1,1anthropic_claude-sonnet-4.6_20260512T072817Z,35,35,0,32,"677,367.0000","46,958.0000","724,325.0000","53,851.0000","778,176.0000","20,695.0000","17,628.0000","1,538.6000",22.8902,30.4264,5.2000,3.2286,0.9143,1.0000,"20,695.0000","24,318.0000",0.0692
4,claude-sonnet-4.6,anthropic/claude-sonnet-4.6,2,2anthropic_claude-sonnet-4.6_20260512T073036Z,35,35,0,31,"671,024.0000","49,851.0000","720,875.0000","54,777.0000","775,652.0000","20,596.4286","18,679.0000","1,565.0571",24.2079,31.7242,5.1714,3.3714,0.8857,1.0000,"20,596.4286","25,021.0323",0.0706
5,claude-sonnet-4.6,anthropic/claude-sonnet-4.6,3,3anthropic_claude-sonnet-4.6_20260512T073516Z,35,35,0,29,"716,510.0000","47,506.0000","764,016.0000","53,126.0000","817,142.0000","21,829.0286","17,657.0000","1,517.8857",23.0164,30.0178,5.1714,3.4000,0.8286,1.0000,"21,829.0286","28,177.3103",0.0650
6,gpt5.5,openai/gpt-5.5,1,openai_gpt-5.5_20260512T075029Z,35,35,0,29,"365,361.0000","34,481.0000","399,842.0000","54,628.0000","454,470.0000","11,424.0571","8,095.0000","1,560.8000",23.5110,31.3405,5.1714,3.0857,0.8286,1.0000,"11,424.0571","15,671.3793",0.1202
7,gpt5.5,openai/gpt-5.5,2,openai_gpt-5.5_20260512T075044Z,35,35,0,29,"305,427.0000","31,945.0000","337,372.0000","53,696.0000","391,068.0000","9,639.2000","8,048.0000","1,534.1714",21.7219,29.2972,5.0286,2.8857,0.8286,1.0000,"9,639.2000","13,485.1034",0.1373
8,gpt5.5,openai/gpt-5.5,3,openai_gpt-5.5_20260512T075850Z,35,35,0,31,"314,468.0000","32,256.0000","346,724.0000","50,847.0000","397,571.0000","9,906.4000","8,127.0000","1,452.7714",22.1376,28.8817,4.2857,2.8571,0.8857,1.0000,"9,906.4000","12,824.8710",0.1279
9,kimi-k2.6,moonshotai/kimi-k2.6,1,1moonshotai_kimi-k2.6_20260512T065725Z,35,33,2,31,"660,050.0000","66,583.0000","726,633.0000","51,959.0000","778,592.0000","20,760.9429","12,228.0000","1,484.5429",75.3714,82.7133,6.6970,4.3030,0.8857,0.9429,"22,019.1818","25,115.8710",0.0667


## Per-Model Summary

In [7]:
models_df = tasks_df.groupby(["model_folder", "model"], dropna=False).agg(
    runs=("run_id", "nunique"),
    tasks=("task_id", "count"),
    completed_tasks=("completed", "sum"),
    failed_tasks=("completed", lambda s: (~s).sum()),
    passed_tasks=("passed", "sum"),
    total_agent_tokens=("agent_total_tokens", "sum"),
    mean_agent_tokens=("agent_total_tokens", "mean"),
    median_agent_tokens=("agent_total_tokens", "median"),
    total_judge_tokens=("judge_total_tokens", "sum"),
    mean_judge_tokens=("judge_total_tokens", "mean"),
    total_combined_tokens=("combined_total_tokens", "sum"),
    mean_combined_tokens=("combined_total_tokens", "mean"),
    average_latency_seconds=("latency", "mean"),
    average_end_to_end_latency_seconds=("end_to_end_latency", "mean"),
    average_tool_calls=("total_tool_calls", "mean"),
    average_steps=("total_steps", "mean"),
    average_tool_appropriateness=("tool_appropriateness", "mean"),
).reset_index()

models_df["pass_rate"] = safe_divide(models_df["passed_tasks"], models_df["tasks"])
models_df["completion_rate"] = safe_divide(models_df["completed_tasks"], models_df["tasks"])
models_df["agent_tokens_per_completed_task"] = safe_divide(models_df["total_agent_tokens"], models_df["completed_tasks"])
models_df["agent_tokens_per_passed_task"] = safe_divide(models_df["total_agent_tokens"], models_df["passed_tasks"])
models_df["combined_tokens_per_completed_task"] = safe_divide(models_df["total_combined_tokens"], models_df["completed_tasks"])
models_df["combined_tokens_per_passed_task"] = safe_divide(models_df["total_combined_tokens"], models_df["passed_tasks"])
models_df["pass_rate_per_100k_combined_tokens"] = safe_divide(models_df["passed_tasks"] * 100_000, models_df["total_combined_tokens"])
models_df["judge_overhead_share"] = safe_divide(models_df["total_judge_tokens"], models_df["total_combined_tokens"])

display(models_df.sort_values("combined_tokens_per_passed_task"))

,model_folder,model,runs,tasks,completed_tasks,failed_tasks,passed_tasks,total_agent_tokens,mean_agent_tokens,median_agent_tokens,total_judge_tokens,mean_judge_tokens,total_combined_tokens,mean_combined_tokens,average_latency_seconds,average_end_to_end_latency_seconds,average_tool_calls,average_steps,average_tool_appropriateness,pass_rate,completion_rate,agent_tokens_per_completed_task,agent_tokens_per_passed_task,combined_tokens_per_completed_task,combined_tokens_per_passed_task,pass_rate_per_100k_combined_tokens,judge_overhead_share
2,gpt5.5,openai/gpt-5.5,3,105,105,0,89,"1,083,938.0000","10,323.2190","8,071.0000","159,171.0000","1,515.9143","1,243,109.0000","11,839.1333",22.4569,29.8398,4.8286,2.9429,0.9168,0.8476,1.0000,"10,323.2190","12,179.0787","11,839.1333","13,967.5169",7.1595,0.1280
0,claude-optus-4.6,anthropic/claude-opus-4.6,3,105,97,8,93,"1,842,097.0000","17,543.7810","15,829.0000","140,170.0000","1,334.9524","1,982,267.0000","18,878.7333",21.6130,27.8625,4.9485,3.1546,0.8449,0.8857,0.9238,"18,990.6907","19,807.4946","20,435.7423","21,314.6989",4.6916,0.0707
1,claude-sonnet-4.6,anthropic/claude-sonnet-4.6,3,105,105,0,92,"2,209,216.0000","21,040.1524","17,820.0000","161,754.0000","1,540.5143","2,370,970.0000","22,580.6667",23.3715,30.7228,5.1810,3.3333,0.7930,0.8762,1.0000,"21,040.1524","24,013.2174","22,580.6667","25,771.4130",3.8803,0.0682
3,kimi-k2.6,moonshotai/kimi-k2.6,3,105,103,2,90,"2,269,475.0000","21,614.0476","14,425.0000","155,822.0000","1,484.0190","2,425,297.0000","23,098.0667",82.1878,89.4704,7.1456,4.1262,0.8466,0.8571,0.9810,"22,033.7379","25,216.3889","23,546.5728","26,947.7444",3.7109,0.0642
5,qwen3.6-plus,qwen/qwen3.6-plus,3,105,105,0,88,"2,423,090.0000","23,077.0476","16,612.0000","146,262.0000","1,392.9714","2,569,352.0000","24,470.0190",52.0183,59.0501,5.9619,4.0762,0.8349,0.8381,1.0000,"23,077.0476","27,535.1136","24,470.0190","29,197.1818",3.4250,0.0569
4,llama4,meta-llama/llama-4-scout,3,105,99,6,7,"1,012,504.0000","9,642.8952","7,515.0000","105,971.0000","1,009.2476","1,118,475.0000","10,652.1429",20.7645,26.3862,1.9293,2.6263,0.8147,0.0667,0.9429,"10,227.3131","144,643.4286","11,297.7273","159,782.1429",0.6259,0.0947


## Tokens Per Successful Task By Answer Type

In [ ]:
answer_type_efficiency = tasks_df.groupby(["model_folder", "model", "answer_type"], dropna=False).agg(
    tasks=("task_id", "count"),
    passed_tasks=("passed", "sum"),
    agent_total_tokens=("agent_total_tokens", "sum"),
    judge_total_tokens=("judge_total_tokens", "sum"),
    combined_total_tokens=("combined_total_tokens", "sum"),
    mean_latency_seconds=("latency", "mean"),
    mean_tool_calls=("total_tool_calls", "mean"),
).reset_index()
answer_type_efficiency["agent_tokens_per_passed_task"] = safe_divide(answer_type_efficiency["agent_total_tokens"], answer_type_efficiency["passed_tasks"])
answer_type_efficiency["combined_tokens_per_passed_task"] = safe_divide(answer_type_efficiency["combined_total_tokens"], answer_type_efficiency["passed_tasks"])
answer_type_efficiency["pass_rate"] = safe_divide(answer_type_efficiency["passed_tasks"], answer_type_efficiency["tasks"])

display(answer_type_efficiency.sort_values(["answer_type", "combined_tokens_per_passed_task"]))

## Task-Level Efficiency

In [ ]:
task_efficiency = tasks_df[[
    "model_folder", "model", "run_index", "run_id", "task_id", "answer_type", "status", "passed",
    "agent_total_tokens", "judge_total_tokens", "combined_total_tokens", "latency", "end_to_end_latency",
    "total_tool_calls", "total_steps", "error_type", "prompt",
]].copy()

display(task_efficiency.sort_values("combined_total_tokens", ascending=False).head(20))

task_rollup = tasks_df.groupby(["model_folder", "model", "task_id", "answer_type"], dropna=False).agg(
    runs=("run_id", "nunique"),
    passed_runs=("passed", "sum"),
    mean_agent_tokens=("agent_total_tokens", "mean"),
    mean_judge_tokens=("judge_total_tokens", "mean"),
    mean_combined_tokens=("combined_total_tokens", "mean"),
    median_combined_tokens=("combined_total_tokens", "median"),
    mean_latency_seconds=("latency", "mean"),
    mean_tool_calls=("total_tool_calls", "mean"),
).reset_index()
task_rollup["pass_rate"] = safe_divide(task_rollup["passed_runs"], task_rollup["runs"])

display(task_rollup.sort_values("mean_combined_tokens", ascending=False).head(20))

## Failure Cost Analysis

In [ ]:
failed_or_not_passed = tasks_df[(~tasks_df["completed"]) | (tasks_df["passed"] == False)].copy()
display(failed_or_not_passed.sort_values("combined_total_tokens", ascending=False)[[
    "model_folder", "model", "run_index", "task_id", "answer_type", "status", "passed", "error_type",
    "agent_total_tokens", "judge_total_tokens", "combined_total_tokens", "latency", "total_tool_calls",
]].head(50))

failure_cost_by_model = failed_or_not_passed.groupby(["model_folder", "model"], dropna=False).agg(
    failed_or_not_passed_tasks=("task_id", "count"),
    agent_tokens_spent=("agent_total_tokens", "sum"),
    judge_tokens_spent=("judge_total_tokens", "sum"),
    combined_tokens_spent=("combined_total_tokens", "sum"),
    mean_combined_tokens=("combined_total_tokens", "mean"),
).reset_index()

display(failure_cost_by_model.sort_values("combined_tokens_spent", ascending=False))

## Comparative Rankings

In [ ]:
ranking_columns = [
    "model_folder", "model", "runs", "tasks", "passed_tasks", "pass_rate", "completion_rate",
    "total_agent_tokens", "total_judge_tokens", "total_combined_tokens",
    "agent_tokens_per_passed_task", "combined_tokens_per_passed_task",
    "pass_rate_per_100k_combined_tokens", "average_end_to_end_latency_seconds", "judge_overhead_share",
]

print("Lowest agent tokens per passed task")
display(models_df[ranking_columns].sort_values("agent_tokens_per_passed_task"))

print("Lowest combined tokens per passed task")
display(models_df[ranking_columns].sort_values("combined_tokens_per_passed_task"))

print("Best pass rate per 100k combined tokens")
display(models_df[ranking_columns].sort_values("pass_rate_per_100k_combined_tokens", ascending=False))

print("Fastest average end-to-end latency")
display(models_df[ranking_columns].sort_values("average_end_to_end_latency_seconds"))

print("Highest judge-token overhead share")
display(models_df[ranking_columns].sort_values("judge_overhead_share", ascending=False))

## Visualizations

In [ ]:
if HAVE_MATPLOTLIB:
    plot_df = models_df.sort_values("combined_tokens_per_passed_task").set_index("model_folder")
    ax = plot_df["combined_tokens_per_passed_task"].plot(kind="bar", figsize=(10, 4), title="Combined Tokens per Passed Task")
    ax.set_xlabel("Model folder")
    ax.set_ylabel("Tokens")
    ax.figure.tight_layout()
else:
    print("Install matplotlib to render notebook plots; all tables and exports are available without it.")

In [ ]:
if HAVE_MATPLOTLIB:
    stacked_df = models_df.set_index("model_folder")[["total_agent_tokens", "total_judge_tokens"]].sort_values("total_agent_tokens")
    ax = stacked_df.plot(kind="bar", stacked=True, figsize=(10, 4), title="Total Agent vs Judge Tokens")
    ax.set_xlabel("Model folder")
    ax.set_ylabel("Tokens")
    ax.figure.tight_layout()
else:
    print("Install matplotlib to render notebook plots; all tables and exports are available without it.")

In [ ]:
if HAVE_MATPLOTLIB:
    ax = models_df.plot(
        kind="scatter",
        x="total_combined_tokens",
        y="pass_rate",
        figsize=(8, 5),
        title="Pass Rate vs Combined Tokens",
    )
    for _, row in models_df.iterrows():
        ax.annotate(row["model_folder"], (row["total_combined_tokens"], row["pass_rate"]), xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("Total combined tokens")
    ax.set_ylabel("Pass rate")
    ax.figure.tight_layout()
else:
    print("Install matplotlib to render notebook plots; all tables and exports are available without it.")

In [ ]:
if HAVE_MATPLOTLIB:
    box_df = tasks_df.pivot(columns="model_folder", values="combined_total_tokens")
    ax = box_df.plot(kind="box", figsize=(10, 4), rot=30, title="Task-Level Combined Token Distribution")
    ax.set_xlabel("Model folder")
    ax.set_ylabel("Tokens")
    ax.figure.tight_layout()
else:
    print("Install matplotlib to render notebook plots; all tables and exports are available without it.")

## Optional: Export Tables

In [9]:
EXPORT_DIR.mkdir(exist_ok=True)

models_df.to_csv(EXPORT_DIR / "cost_model_summary.csv", index=False)
run_summary.to_csv(EXPORT_DIR / "cost_run_summary.csv", index=False)
# tasks_df.to_csv(EXPORT_DIR / "cost_task_runs.csv", index=False)
# task_efficiency.to_csv(EXPORT_DIR / "cost_task_efficiency.csv", index=False)

EXPORT_DIR

WindowsPath('d:/projects/thesis-code/analysis/outputs')